# RAG From Scratch — Interactive Walkthrough
Step-by-step exploration of each pipeline stage.

In [ ]:
import sys
sys.path.insert(0, '..')
from rich import print

## 1. Load a Document

In [ ]:
from src.ingestion.loaders import load_document
doc = load_document('https://en.wikipedia.org/wiki/Retrieval-augmented_generation')
print(doc)

## 2. Chunk It

In [ ]:
from src.ingestion.chunker import chunk_documents
chunks = chunk_documents([doc], strategy='recursive', chunk_size=400, chunk_overlap=50)
print(f'Total chunks: {len(chunks)}')
print(chunks[0])

## 3. Embed

In [ ]:
from src.embeddings.embedder import get_embedding_model
model = get_embedding_model()
vec = model.embed_text('What is RAG?')
print(f'Dim: {len(vec)}, First 5: {vec[:5]}')

## 4. Store in FAISS

In [ ]:
from src.vectorstore.faiss_store import FAISSVectorStore
store = FAISSVectorStore(embedding_model=model)
store.add_chunks(chunks)
print(f'Indexed {len(store)} vectors')

## 5. Retrieve

In [ ]:
results = store.search('What are the components of RAG?', top_k=3)
for r in results:
    print(f'Score: {r.score:.3f} | {r.chunk.content[:100]}...')

## 6. Full Pipeline End-to-End

In [ ]:
from src.pipeline import RAGPipeline
pipeline = RAGPipeline.build()
pipeline.ingest('https://en.wikipedia.org/wiki/Retrieval-augmented_generation')
response = pipeline.query('What is retrieval augmented generation?')
print(response.answer)
print('\nSources:')
for s in response.sources:
    print(f'  - {s["metadata"].get("source")} (score={s["score"]:.3f})')